In [4]:
import sys
from pathlib import Path
raiz = Path.cwd()
while not (raiz / "src").exists() and raiz != raiz.parent:
    raiz = raiz.parent
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
from src.cuantizacion import (cuantizar_dataframe, reconstruir_dataframe,
                              comparar_estadisticas, tabla_resultados)
DIR_TABLAS = raiz / "resultados" / "actividad3" / "tablas"
DIR_TABLAS.mkdir(parents=True, exist_ok=True)

In [5]:
df = pd.read_parquet(raiz / "data" / "dataset_1M.parquet")
df = df.sample(200_000, random_state=0).reset_index(drop=True)
COLS = ["edad", "ingresos", "altura"]
otras = [c for c in df.columns if c not in COLS]
print(df.dtypes)
print("columnas sin cuantizar:", otras)

edad           float64
ingresos       float64
altura         float64
experiencia      int32
categoria       object
dtype: object
columnas sin cuantizar: ['experiencia', 'categoria']


In [6]:
copia = df.copy()
df_cod, modelos = cuantizar_dataframe(df, COLS, "cuantiles", 10)

pd.testing.assert_frame_equal(df, copia)                  # original intacto
assert df_cod.shape == df.shape and list(df_cod.columns) == list(df.columns)
pd.testing.assert_frame_equal(df_cod[otras], df[otras])   # no cuantizadas idénticas
print(df_cod[COLS].dtypes)
df_cod.head()

edad        int16
ingresos    int16
altura      int16
dtype: object


,edad,ingresos,altura,experiencia,categoria
0,2,1,4,15,C
1,1,3,6,14,A
2,3,0,8,18,B
3,9,8,9,6,A
4,0,2,7,14,A


In [7]:
df_rec = reconstruir_dataframe(df_cod, modelos, "media")
est = comparar_estadisticas(df[COLS], df_rec[COLS])
display(est["resumen"].round(4))
display(est["dif_correlacion"].round(5))
print("diferencia máxima de correlación:", est["dif_correlacion_max"])

,columna,dif_media,dif_std,dif_q05,dif_q25,dif_q50,dif_q75,dif_q95
0,edad,-0.0,-0.2486,-1.3122,-0.044,0.0004,0.0461,1.3125
1,ingresos,0.0,-723.6535,-330.7431,-12.979,57.5182,81.3846,4084.7656
2,altura,-0.0,-0.2107,-1.1281,-0.037,-0.0042,0.0119,1.1614


,edad,ingresos,altura
edad,0.00000,0.00179,0.00034
ingresos,0.00179,0.00000,-0.00024
altura,0.00034,-0.00024,0.00000


diferencia máxima de correlación: 0.0017882297482646275
